# **Crossref enrichement exploration**

## **Imports**

In [17]:
import json
import sqlite3

from datetime import UTC, datetime
from urllib.parse import quote

import httpx

from bibliometric_rag_assistant.config import (
    CROSSREF_MAILTO,
    DB_PATH,
    RAW_CROSSREF_DIR,
)

## **Find citations with DOI**

In [18]:
with sqlite3.connect(DB_PATH) as connection:
    connection.row_factory = sqlite3.Row

    rows = connection.execute(
        """
        SELECT
            run_id,
            pmid,
            doi,
            title
        FROM citations
        WHERE run_id = (
            SELECT run_id
            FROM ingestion_runs
            ORDER BY retrieved_at DESC
            LIMIT 1
        )
        AND doi IS NOT NULL
        AND TRIM(doi) <> ''
        LIMIT 5
        """
    ).fetchall()

candidates = [dict(row) for row in rows]

candidates

[{'run_id': '20261016T122907Z',
  'pmid': '37246472',
  'doi': '10.1111/odi.14597',
  'title': '3D-printed temporomandibular joint-mandible combined prosthesis: A prospective study.'},
 {'run_id': '20261016T122907Z',
  'pmid': '37847424',
  'doi': '10.1007/s10006-023-01184-0',
  'title': 'Mandibular reconstruction with TMJ prosthesis: management of osteomyelitis after orthognathic surgery (Case Report).'},
 {'run_id': '20261016T122907Z',
  'pmid': '38071499',
  'doi': '10.1111/joor.13635',
  'title': 'Survival of temporomandibular total joint replacement: A systematic review and meta-analysis.'},
 {'run_id': '20261016T122907Z',
  'pmid': '38685147',
  'doi': '10.1016/j.bjoms.2024.03.003',
  'title': 'Antibiotic prophylaxis in alloplastic temporomandibular joint replacement surgery: A multi-national survey of temporomandibular joint surgeons.'},
 {'run_id': '20261016T122907Z',
  'pmid': '38743036',
  'doi': '10.1097/SCS.0000000000010223',
  'title': 'Surgical Treatment of Temporomandibu

## **Choose a single article**

In [19]:
article = candidates[0]

doi = article["doi"].strip()

print("PMID:", article["pmid"])
print("DOI:", doi)
print("Título:", article["title"])

PMID: 37246472
DOI: 10.1111/odi.14597
Título: 3D-printed temporomandibular joint-mandible combined prosthesis: A prospective study.


## **Consult the crossref API**

In [6]:
CROSSREF_BASE_URL = "https://api.crossref.org"

encoded_doi = quote(doi, safe="")

with httpx.Client(
    timeout=30,
    follow_redirects=True,
    headers={
        "User-Agent": (
            "bibliometric-rag-assistant/0.1 "
            f"({CROSSREF_MAILTO})"
        )
    },
) as client:
    response = client.get(
        f"{CROSSREF_BASE_URL}/works/{encoded_doi}",
        params={"mailto": CROSSREF_MAILTO},
    )

    response.raise_for_status()
    crossref_response = response.json()

In [20]:
response.raise_for_status()

<Response [200 OK]>

## **Explore the response object**

In [9]:
crossref_response

{'status': 'ok',
 'message-type': 'work',
 'message-version': '1.0.0',
 'message': {'indexed': {'date-parts': [[2026, 8, 16]],
   'date-time': '2026-08-16T08:29:30Z',
   'timestamp': 1786868970584,
   'version': 'build-2736575974'},
  'reference-count': 24,
  'publisher': 'Wiley',
  'issue': '3',
  'license': [{'start': {'date-parts': [[2023, 5, 28]],
     'date-time': '2023-05-28T00:00:00Z',
     'timestamp': 1685232000000},
    'content-version': 'vor',
    'delay-in-days': 0,
    'URL': 'http://onlinelibrary.wiley.com/termsAndConditions#vor'}],
  'content-domain': {'domain': ['onlinelibrary.wiley.com'],
   'crossmark-restriction': True},
  'short-container-title': ['Oral Diseases'],
  'published-print': {'date-parts': [[2024, 4]]},
  'abstract': '<jats:title>Abstract</jats:title>\n                  <jats:sec>\n                    <jats:title>Objectives</jats:title>\n                    <jats:p>The study aimed to introduce and evaluate a new customized temporomandibular joint‐mandibl

In [21]:
crossref_response.keys()

dict_keys(['status', 'message-type', 'message-version', 'message'])

The actual metadata comes from the following keys:

In [22]:
crossref_work = crossref_response["message"]

crossref_work.keys()

dict_keys(['indexed', 'reference-count', 'publisher', 'issue', 'license', 'content-domain', 'short-container-title', 'published-print', 'abstract', 'DOI', 'type', 'created', 'page', 'update-policy', 'source', 'is-referenced-by-count', 'title', 'prefix', 'volume', 'author', 'member', 'published-online', 'reference', 'container-title', 'original-title', 'language', 'link', 'deposited', 'score', 'resource', 'subtitle', 'short-title', 'issued', 'references-count', 'journal-issue', 'alternative-id', 'URL', 'archive', 'relation', 'ISSN', 'issn-type', 'subject', 'published', 'assertion'])

In [25]:
crossref_work.get("tile")

In [11]:
fields = [
    "DOI",
    "title",
    "container-title",
    "publisher",
    "type",
    "ISSN",
    "author",
    "published",
    "reference-count",
    "is-referenced-by-count",
]

for field in fields:
    print(f"\n{field}:")
    print(crossref_work.get(field))


DOI:
10.1111/odi.14597

title:
['3D‐printed temporomandibular joint‐mandible combined prosthesis: A prospective study']

container-title:
['Oral Diseases']

publisher:
Wiley

type:
journal-article

ISSN:
['1354-523X', '1601-0825']

author:
[{'given': 'Jisi', 'family': 'Zheng', 'sequence': 'first', 'affiliation': [{'name': "Department of Oral Surgery, Shanghai Ninth People's Hospital Shanghai Jiao Tong University School of Medicine  Shanghai China"}, {'name': 'Shanghai Key Laboratory of Stomatology &amp; Shanghai Research Institute of Stomatology  Shanghai China'}, {'name': 'National Clinical Research Center of Stomatology  Shanghai China'}], 'role': [{'vocabulary': 'crossref', 'role': 'author'}]}, {'given': 'Liang', 'family': 'Huo', 'sequence': 'additional', 'affiliation': [{'name': "Department of Oral Surgery, Shanghai Ninth People's Hospital Shanghai Jiao Tong University School of Medicine  Shanghai China"}, {'name': 'Shanghai Key Laboratory of Stomatology &amp; Shanghai Research In

## **Save the raw response**

In [26]:
retrieved_at = datetime.now(UTC)
timestamp = retrieved_at.strftime("%Y%m%dT%H%M%SZ")

crossref_path = (
    RAW_CROSSREF_DIR
    / f"{timestamp}_{article['pmid']}_crossref.json"
)

crossref_path.write_text(
    json.dumps(
        crossref_response,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Resposta salva em:", crossref_path)

Resposta salva em: /home/lucas/github/bibliometric-rag-assistant/data/raw/crossref/20261016T124411Z_37246472_crossref.json


## **Provisory parsing**

In [27]:
def first_item(values: list | None):
    if not values:
        return None

    return values[0]


def extract_crossref_year(work: dict) -> int | None:
    for field in [
        "published-print",
        "published-online",
        "published",
        "issued",
    ]:
        date_parts = work.get(field, {}).get("date-parts", [])

        if date_parts and date_parts[0]:
            return date_parts[0][0]

    return None


def parse_crossref_work(work: dict) -> dict:
    authors = []

    for author in work.get("author", []):
        name = " ".join(
            part
            for part in [
                author.get("given"),
                author.get("family"),
            ]
            if part
        )

        if name:
            authors.append(name)

    return {
        "doi": work.get("DOI"),
        "title": first_item(work.get("title")),
        "journal": first_item(work.get("container-title")),
        "publisher": work.get("publisher"),
        "document_type": work.get("type"),
        "publication_year": extract_crossref_year(work),
        "issn": work.get("ISSN", []),
        "authors": authors,
        "reference_count": work.get("reference-count"),
        "citation_count": work.get("is-referenced-by-count"),
    }

In [28]:
crossref_record = parse_crossref_work(crossref_work)

crossref_record

{'doi': '10.1111/odi.14597',
 'title': '3D‐printed temporomandibular joint‐mandible combined prosthesis: A prospective study',
 'journal': 'Oral Diseases',
 'publisher': 'Wiley',
 'document_type': 'journal-article',
 'publication_year': 2024,
 'issn': ['1354-523X', '1601-0825'],
 'authors': ['Jisi Zheng',
  'Liang Huo',
  'Zixian Jiao',
  'Xiang Wei',
  'Lingtong Bu',
  'Wenbo Jiang',
  'Yi Luo',
  'Minjie Chen',
  'Chi Yang'],
 'reference_count': 24,
 'citation_count': 6}

In [16]:
article

{'run_id': '20261016T114749Z',
 'pmid': '37246472',
 'doi': '10.1111/odi.14597',
 'title': '3D-printed temporomandibular joint-mandible combined prosthesis: A prospective study.'}

In [29]:
comparison = {
    "pubmed": article,
    "crossref": crossref_record,
}

comparison

{'pubmed': {'run_id': '20261016T122907Z',
  'pmid': '37246472',
  'doi': '10.1111/odi.14597',
  'title': '3D-printed temporomandibular joint-mandible combined prosthesis: A prospective study.'},
 'crossref': {'doi': '10.1111/odi.14597',
  'title': '3D‐printed temporomandibular joint‐mandible combined prosthesis: A prospective study',
  'journal': 'Oral Diseases',
  'publisher': 'Wiley',
  'document_type': 'journal-article',
  'publication_year': 2024,
  'issn': ['1354-523X', '1601-0825'],
  'authors': ['Jisi Zheng',
   'Liang Huo',
   'Zixian Jiao',
   'Xiang Wei',
   'Lingtong Bu',
   'Wenbo Jiang',
   'Yi Luo',
   'Minjie Chen',
   'Chi Yang'],
  'reference_count': 24,
  'citation_count': 6}}